# Truncated Regression

In truncated samples, observations outside the truncation interval are **absent entirely** (not just clipped to a threshold). This happens in survey data when respondents are recruited only above a minimum income, or in administrative data that only records transactions above a reporting threshold.

In [1]:
import numpy as np
import pandas as pd
from censtrunc import TruncatedRegression

rng = np.random.default_rng(7)
n_population = 8000

## Simulate a truncated dataset

We draw a large population, then keep only observations with $0 < Y^* < 2.5$ — about 60% of the population survives the double-sided truncation in this DGP.

In [2]:
X_pop = rng.normal(size=(n_population, 2))
beta_true = np.array([1.0, 0.5, -0.3])
y_star = beta_true[0] + X_pop @ beta_true[1:] + rng.normal(scale=1.0, size=n_population)
L, R = 0.0, 2.5
mask = (y_star > L) & (y_star < R)
X = X_pop[mask]
y = y_star[mask]
print(f'Observed after truncation: {len(y)} / {n_population}')

Observed after truncation: 5741 / 8000


## Fit the truncated regression

In [3]:
model = TruncatedRegression(left=L, right=R).fit(
    X, y, feature_names=['x1', 'x2']
)
print(model.summary())

                              Truncated Regression Results                              
Dep. Variable:           y                   No. Observations:        5741
Model:                   TruncatedRegression Df Model:                3
Method:                  MLE                 Df Residuals:            5737
Date:                    Fri, 29 May 2026    Log-Likelihood:          -4830.9612
Time:                    22:38:24            LL-Null:                 -5136.1192
AIC:                     9669.9225           LLR p-value:             2.962e-133
BIC:                     9696.5440           Pseudo R-squ.:           0.0594
Left truncation:         0                   Right truncation:        2.5
                      coef     std err         z     P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
sigma               0.9944      0.0316   31.5070    0.0000      0.9326      1.0563
const               0.9946      0.0245   40.67

## Comparison: OLS on the truncated sample is biased

Naive OLS on the truncated sample treats the data as if it came from an unrestricted population — leading to biased slope estimates, typically shrunk toward zero.

In [4]:
X_int = np.column_stack([np.ones(len(y)), X])
ols_beta, *_ = np.linalg.lstsq(X_int, y, rcond=None)

comparison = pd.DataFrame({
    'true':         beta_true,
    'OLS (biased)': ols_beta,
    'Truncated MLE': model.coef_,
}, index=['const', 'x1', 'x2'])
comparison['OLS bias']           = comparison['OLS (biased)']  - comparison['true']
comparison['Truncated MLE bias'] = comparison['Truncated MLE'] - comparison['true']
comparison.round(4)

,true,OLS (biased),Truncated MLE,OLS bias,Truncated MLE bias
const,1.0,1.1475,0.9946,0.1475,-0.0054
x1,0.5,0.1952,0.4881,-0.3048,-0.0119
x2,-0.3,-0.1241,-0.3105,0.1759,-0.0105


## Predictions

Two prediction kinds are available for the truncated model:

- `latent`: $X'\hat\beta$ — the population mean
- `truncated`: $\mathbb E[Y | X, L < Y < R]$

In [5]:
preds = pd.DataFrame({
    'observed':  y[:6],
    'latent':    model.predict(X[:6], kind='latent'),
    'truncated': model.predict(X[:6], kind='truncated'),
}).round(3)
preds

,observed,latent,truncated
0,1.590,0.902,1.104
1,0.317,1.137,1.202
2,1.514,1.081,1.178
3,1.376,0.947,1.122
4,1.085,0.764,1.047
5,0.074,0.481,0.936
